# Решения: сигмоида и порог

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('bank_marketing_slim.csv')
df = pd.read_csv(CSV_PATH)
target = (df['y'] == 'yes').astype(int)


In [ ]:
candidate_features = [c for c in df.columns if c != 'y']
feature_columns = [c for c in candidate_features if c != 'duration']
assert 'duration' not in feature_columns


def sigmoid(z):
    arr = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-arr))


vals = np.array([-2.0, 0.0, 2.0])
probs = sigmoid(vals)
sample_proba = np.array([0.11, 0.37, 0.49, 0.51, 0.88])
pred_05 = (sample_proba >= 0.50).astype(int)
pred_03 = (sample_proba >= 0.30).astype(int)
LEAKAGE_NOTE = (
    'Столбец duration нельзя использовать в признаках: это значение известно только после звонка, '
    'поэтому модель с ним не переносится в реальный процесс кампании.'
)
proba_demo = np.array([0.05, 0.16, 0.22, 0.41, 0.61, 0.74, 0.93])
share_04 = float((proba_demo >= 0.4).mean())
share_07 = float((proba_demo >= 0.7).mean())
RULE = 'Перед fit формируем feature_columns и явно исключаем duration; проверяем это через assert.'
RECALL_NOTE = (
    'При ограниченном бюджете обзвона высокий recall помогает не потерять потенциальные yes-кейсы, '
    'если команда готова принять больше ложных срабатываний и потом фильтровать их бизнес-правилами.'
)
print('features:', feature_columns[:8])
print('sigmoid:', probs)
print('pred_05=', pred_05, 'pred_03=', pred_03)
print('share_04=', share_04, 'share_07=', share_07)